# 03 Tabular Baseline Modelling

This notebook trains and evaluates the classical survival baseline models used in the thesis.

The purpose of this notebook is to provide a reproducible baseline comparison for the proposed semi-supervised survival modelling approach.

## Inputs

- processed tabular feature files
- processed survival target files

## Models

- Cox Proportional Hazards model (includes normalization for numerical features)
- Random Survival Forest (includes Hyperparameter Tuning using Random Search 
with hyperparameter space built upon existing work on this dataset and this approach, as a baseline)

## Outputs
- Ablation analysis results (with vs without static features fro each model)
- baseline performance tables 
- saved trained baseline model outputs
- evaluation results for comparison with the proposed model


In [ ]:
# Set project paths and enable imports from src and models directories

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
MODELS_DIR = PROJECT_ROOT / "models"

for path in [SRC_DIR, MODELS_DIR]:
    if str(path) not in sys.path:
        sys.path.append(str(path))

print(f"Notebook location: {Path.cwd()}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Source directory: {SRC_DIR}")
print(f"Models directory: {MODELS_DIR}")

## Import datasets for modelling
### 1. Start with dataset with both dynamic and static features


In [ ]:
from config import (
    X_TRAIN_TRUNC_WS_FILE,
    X_VAL_TRUNC_WS_FILE,
    Y_TRAIN_TRUNC_FILE,
    Y_VAL_TRUNC_FILE
)

In [ ]:
from data_loading import load_processed_training_data

X_train_trunc_with_static, X_val_trunc_with_static, y_train_trunc, y_val_trunc = load_processed_training_data(
    X_TRAIN_TRUNC_WS_FILE,
    X_VAL_TRUNC_WS_FILE,
    Y_TRAIN_TRUNC_FILE,
    Y_VAL_TRUNC_FILE
)

In [ ]:
# retrieve numerical columns for standardization while fitting Cox proportional hazards model
continuous_cols_trunc = [
    col for col in X_train_trunc_with_static.columns 
    if not col.startswith("Spec_") and col != "vehicle_id"
]

In [ ]:
# fit the cox model
from cph import fit_and_evaluate_cph
cph_trunc_with_static = fit_and_evaluate_cph(
    X_train_trunc_with_static,
    y_train_trunc,
    X_val_trunc_with_static,
    y_val_trunc,
    continuous_cols_trunc,
    penalizer=0.1
)

In [ ]:
from config import (
    X_TRAIN_TRUNC_NS_FILE,
    X_VAL_TRUNC_NS_FILE,
    Y_TRAIN_TRUNC_FILE,
    Y_VAL_TRUNC_FILE,
)

In [ ]:
X_train_trunc_no_static, X_val_trunc_no_static, y_train_trunc, y_val_trunc = load_processed_training_data(
    X_TRAIN_TRUNC_NS_FILE,
    X_VAL_TRUNC_NS_FILE,
    Y_TRAIN_TRUNC_FILE,
    Y_VAL_TRUNC_FILE,
)

In [ ]:
# retrieve numerical columns for standardization while fitting Cox proportional hazards model
continuous_cols_trunc_ns = [
    col for col in X_train_trunc_no_static.columns 
    if not col.startswith("Spec_") and col != "vehicle_id"
]

In [ ]:
# fit the cox model
cph_trunc_without_static = fit_and_evaluate_cph(
    X_train_trunc_no_static,
    y_train_trunc,
    X_val_trunc_no_static,
    y_val_trunc,
    continuous_cols_trunc_ns,
    penalizer=0.1
)

## Fitting the Random Survival Forest with random search 
Instead of GridSearch, RandomSearch is preferred for computational reasons. (3-fold cross validation) and the model will be fitted using all features including static specifications. 

In [ ]:
# fit Random Survival Forest model on dynamic features only
from rsf import fit_and_evaluate_rsf_models

rsf_results = fit_and_evaluate_rsf_models(
    X_train=X_train_trunc_no_static,
    y_train=y_train_trunc,
    X_val=X_val_trunc_no_static,
    y_val=y_val_trunc,
    random_state=42,
    n_iter=10,
    cv_splits=3,
    verbose=1
)

print("Baseline validation C-index:", rsf_results["baseline"]["val_c_index"])
print("Tuned validation C-index:", rsf_results["tuned"]["val_c_index"])
print("Tuned best params:", rsf_results["tuned"]["best_params"])

In [ ]:
# RSF with randomized search CV (3 splits) on dynamic+static feature
rsf_results_ws = fit_and_evaluate_rsf_models(
    X_train=X_train_trunc_with_static,
    y_train=y_train_trunc,
    X_val=X_val_trunc_with_static,
    y_val=y_val_trunc,
    random_state=42,
    n_iter=10,
    cv_splits=3,
    verbose=1
)

print("Baseline validation C-index:", rsf_results_ws["baseline"]["val_c_index"])
print("Tuned validation C-index:", rsf_results_ws["tuned"]["val_c_index"])
print("Tuned best params:", rsf_results_ws["tuned"]["best_params"])

### RSF Results Summary (Validation Set)

Two feature configurations were evaluated:

- **Dynamic-only features**
- **Dynamic + static features**

#### Dynamic-only
- Baseline (literature-based): C-index = **0.7253**
- Tuned (restricted randomized search): C-index = **0.7352**

Tuning improved performance in this setting.

#### Dynamic + static
- Baseline (literature-based): C-index = **0.7381**
- Tuned (restricted randomized search): C-index = **0.7321**

The baseline outperformed the tuned model on the validation set.

#### Interpretation
Hyperparameter tuning does not guarantee improved performance on a held-out validation set, as tuning is guided by cross-validation on the training data. In this case, the literature-based baseline appears already well-suited for the dynamic + static feature space.

Static features improve RSF performance overall, as the best result is achieved with dynamic + static features (C-index = **0.7381**).

#### Conclusion
- Use the tuned RSF for the dynamic-only dataset.
- Use the baseline RSF for the dynamic + static dataset.
- Compare these results with Cox and subsequent sequence models under the same feature configurations.

In [ ]:
import pickle
from pathlib import Path

import pandas as pd

RESULTS_DIR = PROJECT_ROOT / "results" / "baselines_trunc"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def save_pickle(obj, path: Path) -> None:
    """
    Save any Python object as a pickle file.
    """
    with open(path, "wb") as f:
        pickle.dump(obj, f)


In [ ]:

def flatten_result_for_table(result: dict, prefix: str = "") -> dict:
    """
    Flatten a nested result dictionary into a one-row table-friendly dictionary.

    Notes
    -----
    - Nested dictionaries are expanded using '<parent>_<child>' keys.
    - Fitted model objects are excluded from the tabular output.
    """
    flat = {}

    for key, value in result.items():
        new_key = f"{prefix}_{key}" if prefix else key

        if key == "fitted_model":
            continue

        if isinstance(value, dict):
            flat.update(flatten_result_for_table(value, prefix=new_key))
        else:
            flat[new_key] = value

    return flat

## Save full results

In [ ]:
# 1. Save full result objects as pickle files
save_pickle(cph_trunc_with_static, RESULTS_DIR / "cph_trunc_with_static.pkl")
save_pickle(cph_trunc_without_static, RESULTS_DIR / "cph_trunc_without_static.pkl")

save_pickle(rsf_results, RESULTS_DIR / "rsf_trunc_no_static.pkl")
save_pickle(rsf_results_ws, RESULTS_DIR / "rsf_trunc_with_static.pkl")

In [ ]:
# -------------------------------------------------------------------
# 2. Save fitted model objects separately as pickle files
# -------------------------------------------------------------------

# Cox models
save_pickle(
    cph_trunc_with_static["fitted_model"],
    RESULTS_DIR / "cph_model_trunc_with_static.pkl"
)
save_pickle(
    cph_trunc_without_static["fitted_model"],
    RESULTS_DIR / "cph_model_trunc_without_static.pkl"
)

# RSF models without static
save_pickle(
    rsf_results["baseline"]["fitted_model"],
    RESULTS_DIR / "rsf_baseline_trunc_no_static.pkl"
)
save_pickle(
    rsf_results["tuned"]["fitted_model"],
    RESULTS_DIR / "rsf_tuned_trunc_no_static.pkl"
)

# RSF models with static
save_pickle(
    rsf_results_ws["baseline"]["fitted_model"],
    RESULTS_DIR / "rsf_baseline_trunc_with_static.pkl"
)
save_pickle(
    rsf_results_ws["tuned"]["fitted_model"],
    RESULTS_DIR / "rsf_tuned_trunc_with_static.pkl"
)



In [ ]:

# -------------------------------------------------------------------
# 3. Build tabular summaries
# -------------------------------------------------------------------

summary_rows = [
    {
        "model_family": "Cox",
        "feature_set": "trunc_with_static",
        **flatten_result_for_table(cph_trunc_with_static)
    },
    {
        "model_family": "Cox",
        "feature_set": "trunc_without_static",
        **flatten_result_for_table(cph_trunc_without_static)
    },
    {
        "model_family": "RSF_baseline",
        "feature_set": "trunc_without_static",
        **flatten_result_for_table(rsf_results["baseline"])
    },
    {
        "model_family": "RSF_tuned",
        "feature_set": "trunc_without_static",
        **flatten_result_for_table(rsf_results["tuned"])
    },
    {
        "model_family": "RSF_baseline",
        "feature_set": "trunc_with_static",
        **flatten_result_for_table(rsf_results_ws["baseline"])
    },
    {
        "model_family": "RSF_tuned",
        "feature_set": "trunc_with_static",
        **flatten_result_for_table(rsf_results_ws["tuned"])
    },
]

summary_df = pd.DataFrame(summary_rows)

In [ ]:
# -------------------------------------------------------------------
# 4. Save tabular summaries
# -------------------------------------------------------------------

summary_df.to_csv(RESULTS_DIR / "baseline_model_summary.csv", index=False)
summary_df.to_pickle(RESULTS_DIR / "baseline_model_summary.pkl")

# Optional: also save Excel for easier manual inspection
summary_df.to_excel(RESULTS_DIR / "baseline_model_summary.xlsx", index=False)


In [ ]:
# -------------------------------------------------------------------
# 5. Print confirmation
# -------------------------------------------------------------------

print("Saved model results and summaries to:")
print(RESULTS_DIR)
summary_df

In [ ]:
# save results and fitted models
import pickle
from pathlib import Path

RESULTS_DIR = PROJECT_ROOT / "results" / "baselines_trunc"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

with open(RESULTS_DIR / "rsf_trunc_ns_random_search_results.pkl", "wb") as f:
    pickle.dump(rsf_no_static, f)



In [ ]:
# Save a summary table of rsf results 
import pandas as pd
rsf_summary = pd.DataFrame([{
    "model": "RSF",
    "n_estimators": 200,
    "min_samples_split": 20,
    "min_samples_leaf": 10,
    "max_features": "sqrt",
    "max_depth": None,
    "cv_c_index": 0.9237,
    "val_c_index": 0.9334
}])

rsf_summary.to_csv(RESULTS_DIR / "rsf_summary.csv", index=False)

In [ ]:
# save the fitted model separately
with open(RESULTS_DIR / "rsf_best_model.pkl", "wb") as f:
    pickle.dump(rsf_no_static["fitted_model"], f)

In [ ]:
# Save validation predictions
best_rsf = rsf_no_static["fitted_model"]
val_risk_scores = best_rsf.predict(X_no_static_val)

with open(RESULTS_DIR / "rsf_val_risk_scores.pkl", "wb") as f:
    pickle.dump(val_risk_scores, f)

In [ ]:
# save a csv version as well
val_predictions_df = pd.DataFrame({
    "risk_score": val_risk_scores,
    "duration": y_val["duration"].values,
    "event": y_val["event"].values
})

val_predictions_df.to_csv(RESULTS_DIR / "rsf_val_predictions.csv", index=False)

In [ ]:
# Save Cox results as well
import pickle
from pathlib import Path
RESULTS_DIR = PROJECT_ROOT / "results" / "baselines"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
with open(RESULTS_DIR / "cox_model_no_static.pkl", "wb") as f:
    pickle.dump(cph_no_static, f)

with open(RESULTS_DIR / "cox_model_with_static.pkl", "wb") as f:
    pickle.dump(cph_with_static, f)

In [ ]:
# save a results dictionary
cox_results_no_static = {
    "model": "CoxPH",
    "val_c_index": cph_no_static["val_c_index"],
    "n_features": X_no_static_train.shape[1],
    "fitted_model": cph_no_static
}

with open(RESULTS_DIR / "cox_results.pkl", "wb") as f:
    pickle.dump(cox_results_no_static, f)

In [ ]:
cox_results_with_static = {
    "model": "CoxPH",
    "val_c_index": cph_with_static["val_c_index"],
    "n_features": X_with_static_train.shape[1],
    "fitted_model": cph_with_static
}

with open(RESULTS_DIR / "cox_results_with_static.pkl", "wb") as f:
    pickle.dump(cox_results_with_static, f)

In [ ]:
# save as table
import pandas as pd
cox_summary_no_static = pd.DataFrame([{
    "model": cph_no_static["model"],
    "penalizer": cph_no_static["penalizer"],
    "val_c_index": cph_no_static["val_c_index"],
    "n_features": X_no_static_train.shape[1]
}])

cox_summary_no_static.to_csv(RESULTS_DIR / "cox_summary_no_static.csv", index=False)

In [ ]:
cox_summary_with_static = pd.DataFrame([{
    "model": cph_with_static["model"],
    "penalizer": cph_with_static["penalizer"],
    "val_c_index": cph_with_static["val_c_index"],
    "n_features": X_with_static_train.shape[1]
}])

cox_summary_with_static.to_csv(RESULTS_DIR / "cox_summary_with_static.csv", index=False)

In [ ]:
# Save validation predictions
# Without static, save as csv
cox_val_risk_scores_no_static = pd.DataFrame({
    "risk_score": cph_no_static["partial_hazards"].values.ravel(),
    "duration": y_val["duration"].values,
    "event": y_val["event"].values
})
cox_val_risk_scores_no_static.to_csv(
    RESULTS_DIR / "cox_val_predictions_no_static.csv",
    index=False
)
# with static, save as csv
cox_val_risk_scores_with_static = pd.DataFrame({
    "risk_score": cph_with_static["partial_hazards"].values.ravel(),
    "duration": y_val["duration"].values,
    "event": y_val["event"].values
})
cox_val_risk_scores_with_static.to_csv(
    RESULTS_DIR / "cox_val_predictions_with_static.csv",
    index=False
)

In [ ]:
# save fitted model separately
with open(RESULTS_DIR / "cox_best_model_no_static.pkl", "wb") as f:
    pickle.dump(cph_no_static["fitted_model"], f)

In [ ]:
# save the scaler
with open(RESULTS_DIR / "cox_scaler_no_static.pkl", "wb") as f:
    pickle.dump(cph_no_static["scaler"], f)